In [2]:
import pandas as pd
import numpy as np
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest

# -----------------------------------------------
# 1. Работа с датасетом diy.txt
# -----------------------------------------------
print("="*60)
print("Задание: diy.txt – LOF и Isolation Forest")
print("="*60)

# Загрузка данных
diy_df = pd.read_csv('diy.txt')
print("Размер diy.txt:", diy_df.shape)
print(diy_df.head())

# Используем только Frequency и Monetary_A, фильтр Frequency > 1
diy_filtered = diy_df[diy_df['Frequency'] > 1].copy()
X_diy = diy_filtered[['Frequency', 'Monetary_A']].values

print(f"Количество строк с Frequency > 1: {len(X_diy)}")

# LOF (novelty=False, параметры по умолчанию)
lof_diy = LocalOutlierFactor(novelty=False, contamination='auto')
y_pred_lof_diy = lof_diy.fit_predict(X_diy)
anomalies_lof_diy = np.sum(y_pred_lof_diy == -1)
print(f"\nLOF (обычный режим) число аномалий: {anomalies_lof_diy}")

# Isolation Forest (параметры по умолчанию)
iso_diy = IsolationForest(contamination='auto', random_state=42)
y_pred_iso_diy = iso_diy.fit_predict(X_diy)
anomalies_iso_diy = np.sum(y_pred_iso_diy == -1)
print(f"Isolation Forest число аномалий: {anomalies_iso_diy}")

# Пересечение (для справки)
common = np.sum((y_pred_lof_diy == -1) & (y_pred_iso_diy == -1))
print(f"Общих аномалий (LOF & IF): {common}")

# -----------------------------------------------
# 2. Работа с датасетом banks.txt
# -----------------------------------------------
print("\n" + "="*60)
print("Задание: banks.txt – LOF и Isolation Forest")
print("="*60)

# Загрузка данных с правильной кодировкой
diy_df = pd.read_csv('diy.txt', encoding='cp1251')
banks_df = pd.read_csv('banks.txt', encoding='cp1251')

# Отделим названия банков (столбец Bank) и числовые признаки
X_banks = banks_df.drop(columns=['Bank']).values

# -------------------------------------------------
# п.1 LOF в обычном режиме (novelty=False)
# -------------------------------------------------
lof_banks = LocalOutlierFactor(novelty=False, contamination='auto')
y_pred_lof_banks = lof_banks.fit_predict(X_banks)
anomalies_lof1 = np.sum(y_pred_lof_banks == -1)
print(f"\n1. LOF (обычный режим) число аномалий: {anomalies_lof1}")

# -------------------------------------------------
# п.2 Число аномалий уже выведено
# -------------------------------------------------

# -------------------------------------------------
# п.3 Отфильтруем аномалии и обучим LOF с novelty=True
# -------------------------------------------------
inlier_mask = y_pred_lof_banks == 1
X_inliers = X_banks[inlier_mask]
print(f"Количество не-аномалий (для обучения novelty LOF): {len(X_inliers)}")

# LOF в режиме обнаружения новизны
lof_novelty = LocalOutlierFactor(novelty=True, contamination='auto')
lof_novelty.fit(X_inliers)

# -------------------------------------------------
# п.4 Прогоним ранее отфильтрованные аномалии через модель п.3
# -------------------------------------------------
outliers = X_banks[~inlier_mask]   # аномалии, найденные в п.1
if len(outliers) > 0:
    pred_novelty = lof_novelty.predict(outliers)
    # Сколько из них снова распознаны как аномалии ( -1)
    coincidences = np.sum(pred_novelty == -1)
    print(f"\n4. Число совпадений (аномалии п.1, снова найденные novelty‑LOF): {coincidences} из {len(outliers)}")
else:
    print("\n4. Аномалий не найдено в п.1, проверка невозможна.")

# -------------------------------------------------
# п.5 Решение задачи методом Isolation Forest
# -------------------------------------------------
iso_banks = IsolationForest(contamination='auto', random_state=42)
y_pred_iso_banks = iso_banks.fit_predict(X_banks)
anomalies_iso_banks = np.sum(y_pred_iso_banks == -1)
print(f"\n5. Isolation Forest число аномалий: {anomalies_iso_banks}")

# Сравнение LOF (обычный) и Isolation Forest
common_banks = np.sum((y_pred_lof_banks == -1) & (y_pred_iso_banks == -1))
print(f"   Общих аномалий (LOF & IF): {common_banks}")

# -------------------------------------------------
# п.6 Изучение метрик качества (для задач anomaly detection)
# -------------------------------------------------
print("\n6. Метрики качества (без эталонных меток)")

# Так как истинные метки неизвестны, можно вычислить метрики,
# считая один из детекторов «референсным». Например, возьмём LOF как истину.
# Вычислим precision, recall, F1 для Isolation Forest относительно LOF.
tp = common_banks
fp = anomalies_iso_banks - tp
fn = anomalies_lof1 - tp
if tp + fp > 0:
    precision = tp / (tp + fp)
else:
    precision = 0.0
if tp + fn > 0:
    recall = tp / (tp + fn)
else:
    recall = 0.0
if precision + recall > 0:
    f1 = 2 * precision * recall / (precision + recall)
else:
    f1 = 0.0

print(f"   Если считать LOF эталоном, то для Isolation Forest:")
print(f"   Precision = {precision:.3f}, Recall = {recall:.3f}, F1 = {f1:.3f}")

# Дополнительно: можно оценить согласованность через индекс Жаккара
if anomalies_lof1 + anomalies_iso_banks - common_banks > 0:
    jaccard = common_banks / (anomalies_lof1 + anomalies_iso_banks - common_banks)
else:
    jaccard = 0.0
print(f"   Индекс Жаккара (пересечение/объединение) = {jaccard:.3f}")

print("\nЗадание выполнено.")

Задание: diy.txt – LOF и Isolation Forest
Размер diy.txt: (42746, 5)
      ClientID  Recency  Frequency  Monetary_Q  Monetary_A
0  client13166      682          2          23        2705
1   client1239       35         43         219       42161
2  client30041      190         25         133       16057
3  client36276      289          4          12        4614
4  client14136      217          6          36       35870
Количество строк с Frequency > 1: 29887

LOF (обычный режим) число аномалий: 141
Isolation Forest число аномалий: 4413
Общих аномалий (LOF & IF): 119

Задание: banks.txt – LOF и Isolation Forest

1. LOF (обычный режим) число аномалий: 56
Количество не-аномалий (для обучения novelty LOF): 315

4. Число совпадений (аномалии п.1, снова найденные novelty‑LOF): 56 из 56

5. Isolation Forest число аномалий: 30
   Общих аномалий (LOF & IF): 30

6. Метрики качества (без эталонных меток)
   Если считать LOF эталоном, то для Isolation Forest:
   Precision = 1.000, Recall = 0.536, 